In [25]:
import os
import time
import random
import sqlite3
import pandas as pd
from bs4 import BeautifulSoup as soup
from splinter import Browser
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from webdriver_manager.core.driver_cache import DriverCacheManager


In [ ]:
class ImdbScraper:
    def __init__(self, base_url, year):
        self.base_url = base_url
        self.results  = []
        self.year     = year
        self.db_path  = os.path.join('..', 'database', 'imdb_movies.db')
        self.BATCH_SIZE = 50

    # ------------------------------------------------------------------ #
    #  Browser setup                                                       #
    # ------------------------------------------------------------------ #
    def _init_browser(self):
        project_root = os.path.dirname(os.getcwd())
        cache_path   = os.path.join(project_root, "drivers")
        os.makedirs(cache_path, exist_ok=True)

        cache_manager = DriverCacheManager(root_dir=cache_path)
        driver_path   = ChromeDriverManager(cache_manager=cache_manager).install()
        service       = Service(driver_path)

        chrome_options = Options() 
        chrome_options.add_argument("--disable-gpu")
        chrome_options.add_argument("--no-sandbox")
        chrome_options.add_argument("--disable-dev-shm-usage") 
        chrome_options.add_argument("--disable-blink-features=AutomationControlled")
        chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
        chrome_options.add_experimental_option("useAutomationExtension", False)

        return Browser('chrome', service=service, options=chrome_options, headless=False)
    # ------------------------------------------------------------------ #
    #  Helpers                                                             #
    # ------------------------------------------------------------------ #
    def _dismiss_overlays(self, browser):
        """Closes any cookie banners or popups that block clicks."""
        dismiss_xpaths = [
            '//button[@data-testid="accept-button"]',
            '//button[contains(text(), "Accept")]',
            '//button[contains(text(), "Reject All")]',
            '//div[@data-testid="consent-overlay"]//button',
        ]
        for xpath in dismiss_xpaths:
            try:
                btn = WebDriverWait(browser.driver, 3).until(
                    EC.element_to_be_clickable((By.XPATH, xpath))
                )
                btn.click()
                print("✅ Dismissed overlay")
                time.sleep(1)
                break
            except:
                continue

    def _save_to_db(self, results):
        """Saves a list of dicts to the database."""
        if not results:
            return
        conn = sqlite3.connect(self.db_path)
        pd.DataFrame(results).to_sql('movies', conn, if_exists='append', index=False)
        conn.close()

    def _already_scraped(self):
        """Returns a set of (year, month) tuples already in the database."""
        if not os.path.exists(self.db_path):
            return set()
        try:
            conn     = sqlite3.connect(self.db_path)
            existing = pd.read_sql("SELECT DISTINCT year, month FROM movies", conn)
            conn.close()
            return set(zip(existing['year'].astype(str), existing['month'].astype(str)))
        except:
            return set()

    # ------------------------------------------------------------------ #
    #  Parsers                                                             #
    # ------------------------------------------------------------------ #
    def get_title_and_link(self, container):
        tag = container.find('a', class_='ipc-title-link-wrapper')
        if tag:
            name = tag.find('h3').text.strip()
            if ". " in name:
                name = name.split(". ", 1)[1]
            link = "https://www.imdb.com" + tag.get('href').split('?')[0]
            return name, link
        return "N/A", "N/A"

    def get_metadata(self, container):
        """Extracts Year and Duration from li tags."""
        year     = "N/A"
        duration = "N/A"
        for li in container.find_all('li', class_='ipc-inline-list__item'):
            text = li.get_text(strip=True)
            if text.isdigit() and len(text) == 4:
                year = text
            elif ('h' in text or 'm' in text) and any(c.isdigit() for c in text):
                if len(text) < 10:
                    duration = text
        return year, duration

    def get_metascore(self, container):
        score_tag = container.find('span', class_='metacritic-score-box')
        if not score_tag:
            score_tag = container.find('span', style=lambda s: s and 'background-color' in s)
        return score_tag.text.strip() if score_tag else "N/A"

    def get_storyline(self, container):
        story_tag = container.find('div', class_='ipc-html-content-inner-div')
        return story_tag.text.strip() if story_tag else "N/A"

    def get_ratings(self, container):
        rating_tag = container.find('span', class_='ipc-rating-star--rating')
        rating     = rating_tag.text.strip() if rating_tag else "N/A"
        votes_tag  = container.find('span', class_='ipc-rating-star--voteCount')
        votes      = votes_tag.text.strip().replace('(', '').replace(')', '').strip() if votes_tag else "N/A"
        return rating, votes

    # ------------------------------------------------------------------ #
    #  Core scraping                                                       #
    # ------------------------------------------------------------------ #
    def scrape_data(self, browser, start_limit, end_limit):
        url = (
            f"https://www.imdb.com/search/title/"
            f"?title_type=feature"
            f"&release_date={start_limit}-01,{end_limit}-31"
            f"&sort=num_votes,desc"
        )
        month_num = int(start_limit.split('-')[1])

        print(f"\n📥 Visiting: {url}")
        browser.visit(url)
        time.sleep(5)

        self._dismiss_overlays(browser)

        print(f"📥 Loading all results for {start_limit}...")
        while True:
            try:
                browser.execute_script("window.scrollTo(0, document.body.scrollHeight);")
                time.sleep(random.uniform(3, 5))
                load_more = browser.find_by_xpath("//button[contains(@class, 'ipc-see-more__button')]")
                if load_more and load_more.visible:
                    browser.execute_script("arguments[0].click();", load_more._element)
                    print("  - Loading next 50 items...")
                    time.sleep(random.uniform(3, 5))
                else:
                    break
            except:
                break

        print("  ⏬ Scrolling to render all items...")
        total_height = browser.execute_script("return document.body.scrollHeight")
        current = 0
        while current < total_height:
            browser.execute_script(f"window.scrollTo(0, {current});")
            current     += 2000
            total_height = browser.execute_script("return document.body.scrollHeight")
        time.sleep(1)

        page_soup      = soup(browser.html, 'html.parser')
        all_containers = page_soup.find_all('div', class_='ipc-metadata-list-summary-item__c')
        total_found    = len(all_containers)  
        print(f"  📦 Found {total_found} containers")

        batch      = []
        saved      = 0   
        skipped    = 0   

        for item in all_containers:
            name, link    = self.get_title_and_link(item)
            year_val, dur = self.get_metadata(item)
            imdb, votes   = self.get_ratings(item)
            storyline     = self.get_storyline(item)
            metascore     = self.get_metascore(item)

            if storyline == 'N/A' or not storyline.strip():
                skipped += 1   
                continue

            batch.append({
                "movie_name"  : name,
                "year"        : year_val,
                "month"       : month_num,
                "duration"    : dur,
                "imdb_rating" : imdb,
                "vote_count"  : votes,
                "metascore"   : metascore,
                "storyline"   : storyline,
                "url"         : link
            })
            saved += 1

            if len(batch) == self.BATCH_SIZE:
                self._save_to_db(batch)
                batch = []
 
        if batch:
            self._save_to_db(batch) 
            
        print(f"✅ Saved {saved} | ⏭️ Skipped {skipped} | 📦 Total {total_found} — {start_limit}")

        del page_soup
        del all_containers
        del batch
        
    # ------------------------------------------------------------------ #
    #  Entry point                                                         #
    # ------------------------------------------------------------------ #
    def start_scraping(self):
        already_scraped = self._already_scraped()
        browser         = self._init_browser()
        months          = pd.period_range(start=self.year, periods=12, freq='M')

        for i in range(len(months)):
            m_str     = str(months[i])
            year_num  = m_str.split('-')[0]
            month_num = str(int(m_str.split('-')[1]))

            if (year_num, month_num) in already_scraped:
                print(f"⏭️  Skipping {m_str} — already in database")
                continue

            self.scrape_data(browser, m_str, m_str)

        browser.quit()
        print("\n✅ Scraping complete!")

In [ ]:
# No hardcoded dates in the URL — scraper fills them dynamically
url = 'https://www.imdb.com/search/title/?title_type=feature&sort=num_votes,desc'

ob = ImdbScraper(url, 2001)
ob.start_scraping() 


📥 Visiting: https://www.imdb.com/search/title/?title_type=feature&release_date=2001-01-01,2001-01-31&sort=num_votes,desc
📥 Loading all results for 2001-01...
  ⏬ Scrolling to render all items...
  📦 Found 0 containers
✅ Saved 0 | ⏭️ Skipped 0 | 📦 Total 0 — 2001-01

📥 Visiting: https://www.imdb.com/search/title/?title_type=feature&release_date=2001-02-01,2001-02-31&sort=num_votes,desc
📥 Loading all results for 2001-02...
  ⏬ Scrolling to render all items...
  📦 Found 0 containers
✅ Saved 0 | ⏭️ Skipped 0 | 📦 Total 0 — 2001-02

📥 Visiting: https://www.imdb.com/search/title/?title_type=feature&release_date=2001-03-01,2001-03-31&sort=num_votes,desc
📥 Loading all results for 2001-03...
  ⏬ Scrolling to render all items...
  📦 Found 0 containers
✅ Saved 0 | ⏭️ Skipped 0 | 📦 Total 0 — 2001-03

📥 Visiting: https://www.imdb.com/search/title/?title_type=feature&release_date=2001-04-01,2001-04-31&sort=num_votes,desc
📥 Loading all results for 2001-04...
  ⏬ Scrolling to render all items...
  📦 Fo

In [22]:
def clean_db(db_path):
    """Pass any .db file path — removes all rows where storyline is missing."""
    if not os.path.exists(db_path):
        print(f"⚠️ File not found: {db_path}")
        return

    conn   = sqlite3.connect(db_path)
    before = pd.read_sql("SELECT COUNT(*) as count FROM movies", conn).iloc[0]['count']

    conn.execute("""
        DELETE FROM movies
        WHERE storyline = 'N/A'
           OR storyline IS NULL
           OR TRIM(storyline) = ''
    """)
    conn.commit()

    after = pd.read_sql("SELECT COUNT(*) as count FROM movies", conn).iloc[0]['count']
    conn.close()

    print(f"🧹 Removed {before - after} rows — {after} rows remaining in {db_path}")


# usage — call it anytime with your db path
clean_db(r"D:\IMP  ML  PROJECTS\Mood2Movie\database\imdb_movies.db")

🧹 Removed 2072 rows — 3360 rows remaining in D:\IMP  ML  PROJECTS\Mood2Movie\database\imdb_movies.db
